# Gulfstream walkthrough — YCS via panelyzer features

Same **Part A / Part B** story as `01_ycs_zero_rates_workflow.ipynb`, but features
come from **panelyzer** (`feature_builder.create_features`) instead of
`generate_yield_features`.

| Part | Focus |
|------|--------|
| A | PCA → Graph 1 + Graph 2 |
| B | Kernel PCA → Graph 1 + Graph 2 |

**Source:** `config/sources/notebook_ycs_panelyzer.yaml`  
**Expressions:** `config/features/ycs_panelyzer_subset.yaml` (subset of slopes / flies / 10y spreads)

User-supplied yield curve (+ FX) DuckDB window; panelyzer builds the detection panel.


## 0. Project setup


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
import polars as pl
from plotnine import aes, facet_wrap, geom_line, ggplot, labs, theme, theme_bw

NOTEBOOK_DIR = Path.cwd()
if (NOTEBOOK_DIR / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "pyproject.toml").exists():
    ROOT = NOTEBOOK_DIR.parent
else:
    raise FileNotFoundError("Run from the gulfstream repo (or notebooks/).")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

OUT_DIR = ROOT / "outputs" / "notebooks" / "ycs_panelyzer"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT =", ROOT)
print("OUT_DIR =", OUT_DIR)


## 1. Load through the public API (panelyzer features)

`load_features` reads `notebook_ycs_panelyzer.yaml`, which sets

`feature_generator: gulfstream.data.feature_generation.generate_panelyzer_features`

with `feature_generator_kwargs.config` pointing at the expression YAML.


In [ ]:
from gulfstream import load_features
from gulfstream.common import frames

SOURCE_YAML = ROOT / "config" / "sources" / "notebook_ycs_panelyzer.yaml"
features_df = load_features(SOURCE_YAML, project_root=ROOT)
print("shape:", features_df.shape, "n_features:", frames.n_features(features_df))
print("date range:", features_df["date"].min(), "→", features_df["date"].max())
print("features:", frames.feature_columns(features_df))
features_df.head(3)


## 2. Explore a few panelyzer series


In [ ]:
plot_cols = [
    c
    for c in [
        "USA_Y010p0",
        "DEU_Y010p0",
        "ITA_Y010p0",
        "USA_slope_30_2",
        "USA_fly_2_10_30",
        "USA_DEU_Y010p0_spd",
    ]
    if c in features_df.columns
]
long = (
    features_df.select(["date", *plot_cols])
    .unpivot(index="date", on=plot_cols, variable_name="series", value_name="value")
    .to_pandas()
)
long["date"] = pd.to_datetime(long["date"])

(
    ggplot(long, aes("date", "value", color="series"))
    + geom_line(size=0.4)
    + facet_wrap("~series", scales="free_y", ncol=1)
    + theme_bw()
    + theme(figure_size=(10, 2.0 * max(len(plot_cols), 1)))
    + labs(title="Panelyzer YCS subset features", x="", y="")
)


## 3. Shared helpers (public API)

Graph 1 via `run_single_segmentation`; Graph 2 via `refine_regimes`.


In [ ]:
import copy
from IPython.display import Image, display

from gulfstream import (
    plot_regimes,
    refine_regimes,
    regime_intervals,
    run_single_segmentation,
    seed_regimes_from_results,
)
from gulfstream.common import frames, utils


def load_core_params(img_dir: Path) -> dict:
    params = utils.read_config_yaml(
        str(ROOT / "config" / "graph1" / "default_core.yaml"),
        img_dir=str(img_dir),
        log_dir=str(ROOT / "outputs" / "logs"),
    )
    params["test_num"] = 0
    params["metrics"]["mode"] = "display_and_write"
    params["metrics"]["plot"] = True
    params["metrics"]["dir"] = str(img_dir)
    params["metrics"]["image_dir"] = str(img_dir)
    params["robustness"]["enabled"] = False
    params["stability"]["enabled"] = False
    return params


def with_dimred(params: dict, method: str) -> dict:
    out = copy.deepcopy(params)
    method = method.lower()
    out["algo"]["dimred"] = [method]
    if method == "kpca":
        out["algo"]["kpca_kernel_params"] = [{"kernel": "rbf", "gamma": "median"}]
    elif method != "pca":
        raise ValueError(f"Unsupported dimred for this notebook: {method}")
    return out


def summarize(res, label: str, df: pl.DataFrame) -> None:
    dates = frames.dates_series(df).to_list()
    print(f"[{label}] kept={res.bkpts}  invalid={res.invalid_bkpts}")
    for b in res.bkpts:
        print(f"  bkpt {b} → {dates[b]}")


def run_g1(df: pl.DataFrame, params: dict, label: str, plot_vars: list[str]):
    print(f"=== Graph 1 · {label} · dimred={params['algo']['dimred']} ===")
    proc = run_single_segmentation(df, params)
    summarize(proc, label, df)
    plot_regimes(df, proc, variables=plot_vars[:2], title=f"Graph 1 · {label}", mode="display")
    return proc


def run_g2(
    df: pl.DataFrame,
    params: dict,
    seed_res,
    out_dir: Path,
    label: str,
    plot_vars: list[str],
    *,
    max_iter: int = 3,
    threshold: float = 1e-6,
) -> Path:
    out_dir.mkdir(parents=True, exist_ok=True)
    g2 = copy.deepcopy(params)
    g2["metrics"]["dir"] = str(out_dir)
    g2["metrics"]["image_dir"] = str(out_dir)
    g2["metrics"]["mode"] = "display_and_write"
    g2["metrics"]["plot"] = True
    g2["retrain"] = {
        "interactive": False,
        "features": ["__auto__"],
        "num_worst_features": min(5, frames.n_features(df)),
        "threshold": threshold,
        "max_iter": max_iter,
        "score_method": "mse_to_mean",
        "score": {},
        "regimes_df": None,
    }
    print(f"=== Graph 2 · {label} · seeding from Graph 1 ===")
    print(seed_regimes_from_results(df, seed_res).to_dicts())
    refined = refine_regimes(df, g2, seed=seed_res)
    if refined is not None:
        summarize(refined, f"{label} Graph 2", df)
        plot_regimes(
            df, refined, variables=plot_vars[:2], title=f"Graph 2 · {label}", mode="display"
        )
    for p in sorted(out_dir.rglob("retrain_iteration_*.png"))[:6]:
        print(" ", p.relative_to(out_dir))
        try:
            display(Image(filename=str(p)))
        except Exception as exc:
            print("  (could not display)", exc)
    return out_dir


print("Helpers ready")


---
# Part A — PCA (baseline)

Default core: **PCA → RFF → PELT → MMD**, then Graph 2 seeded from that run.


## A.1 Graph 1 (PCA + PELT)


In [ ]:
params_pca = with_dimred(load_core_params(OUT_DIR / "pca"), "pca")
params_pca["metrics"]["features_to_plot"] = plot_cols[:3]
proc_pca = run_g1(features_df, params_pca, "PCA", plot_cols)
regime_intervals(proc_pca, frames.dates_series(features_df).to_list())


## A.2 Graph 2 (seeded from PCA)

Auto-retrain loop with default `mse_to_mean` score heatmap.


In [ ]:
g2_pca_dir = run_g2(
    features_df, params_pca, proc_pca, OUT_DIR / "pca" / "graph2", "PCA", plot_cols, max_iter=3
)


---
# Part B — Kernel PCA


## B.1 Graph 1 (kPCA)


In [ ]:
params_kpca = with_dimred(load_core_params(OUT_DIR / "kpca"), "kpca")
params_kpca["metrics"]["features_to_plot"] = plot_cols[:3]
proc_kpca = run_g1(features_df, params_kpca, "kPCA", plot_cols)


## B.2 Graph 2 (seeded from kPCA)


In [ ]:
g2_kpca_dir = run_g2(
    features_df, params_kpca, proc_kpca, OUT_DIR / "kpca" / "graph2", "kPCA", plot_cols, max_iter=3
)


## CLI equivalents

```bash
uv run python -m gulfstream.cli --mode graph1 \
  --config config/graph1/default_core.yaml \
  --source-config config/sources/notebook_ycs_panelyzer.yaml

uv run python -m gulfstream.cli --mode graph2 \
  --config config/graph2/full_graph2.yaml \
  --source-config config/sources/notebook_ycs_panelyzer.yaml
```

Edit expressions in `config/features/ycs_panelyzer_subset.yaml` without changing the loader.
